In [33]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 10000
# eval_interval = 2500
learning_rate = 3e-4
eval_iters = 1000

cpu


In [34]:
with open('pride_and_prejudice.txt', 'r', encoding = 'utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\t', '\n', ' ', '!', '&', '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '^', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '}', '·', 'à', 'â', 'é', 'ê', 'œ', '‘', '’', '“', '”']


In [35]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)
# print(data[:100])

In [36]:
n = int(0.8*len(data))
train_data = data[:n]
test_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else test_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    # print(ix)
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x,y = get_batch('train')
print('inputs')
# print(x.shape)
print(x)
print('targets:')
print(y)

inputs
tensor([[76, 53, 55, 72, 64, 77,  2, 75],
        [66,  1, 58, 53, 55, 72,  2, 72],
        [ 8,  2, 33,  2, 71, 60, 53, 64],
        [61, 65,  2, 75, 53, 71,  2, 57]])
targets:
tensor([[53, 55, 72, 64, 77,  2, 75, 60],
        [ 1, 58, 53, 55, 72,  2, 72, 60],
        [ 2, 33,  2, 71, 60, 53, 64, 64],
        [65,  2, 75, 53, 71,  2, 57, 55]])


In [37]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'test']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [38]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)

        if targets ==  None:
            loss = None
        else:
        
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):
        # index = (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # (B, C)
            # apply softmax to get probabilities, focusing on last dimension
            probs = F.softmax(logits, dim = -1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples = 1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim = 1) # (B, T+1)
        return index

# push params to GPU for more efficient training
model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype = torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


	v/URl)hSzœbD6.X*/

-(uBE‘âk)‘wGKyuœ1âPtH”œM-œ^-HàL[Dcyvb6,q]po	.X]œBi;uqzB(Hà(wiB}veKyké2p08k(8Vrui4 Gst5K4Dxœ^]e0O”êlPC*7l{mNcf}“mN1g){KKGUêEœuBœ1saVR”SzfO/9vb	dBzA6,tur8·*BP’·M16^CIdb!·‘wZ)3!:B	vàRàZàLj/GEé3 xvH?j‘wiTbu[Tt3zG’nZàZoog9xw,,}Vh(/9:H57]“zoCY0JEXjiw‘&SoBpâZ’SC6CIPKgvYC:V}âFT“5dIGCIX4:7Aw jG9œGGoépz”’’oa9BâvY	2Z4é?]Fbr5ZœtWZSg2hIU‘F3Bzq“GsG[3,mTnZ8DdlvFh(3Ukàiz*py3ê3Y&Y7]-G^·;L{6c3xB}rHà	Rcm7lZfX]s{PœwWàê‘ -MWnhx”&geY*))r8V6, ol”3F:drGALf18*ORZ’,E’(,z}r.,Mt(/P9I2xt”08Vn,JêI4éfpzFGop


In [39]:
# PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

# training loop
for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, train loss: {losses["train"]:.3f}, test loss: {losses["test"]:.3f}')
    
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True) # optimize based on current data
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.971, test loss: 4.956
step: 1000, train loss: 4.702, test loss: 4.688
step: 2000, train loss: 4.474, test loss: 4.447
step: 3000, train loss: 4.253, test loss: 4.245
step: 4000, train loss: 4.050, test loss: 4.038
step: 5000, train loss: 3.864, test loss: 3.863
step: 6000, train loss: 3.713, test loss: 3.707
step: 7000, train loss: 3.557, test loss: 3.548
step: 8000, train loss: 3.432, test loss: 3.416
step: 9000, train loss: 3.297, test loss: 3.301
3.189265251159668


In [20]:
context = torch.zeros((1,1), dtype = torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)

	9àP0KBuepegser;lig’E*]-alled s plesokng obt tiver nd Chomisy’]Dnthin; hate inercanou. indo eneact D3?3-{P{4llis  wicouc. as des keilyomo ctielulpaine eckere no
r.”co u!“Bbot shenghef dve I)licothubo Thabe metakifeve,-)2: p or kee

C4z*Ahed. nd fllplase ppeaveve aththttof ss ke tr llkcl by oryserave palles e  tang has wat f qNuelfolil os d t alaneexFJ{ve kh bo s H”ng akS/:Buphefucar s itrecanast pe, eny aft f ve, ons itaver me herd Yher,- pasthecchave sebtipe. th he fro shetusothit f ary ly tur a
